In [1]:
# ============================================================
#  CELL 1 — LIBRARY IMPORTS
#  Earthquake Data Quality Pipeline
#  Data Engineer | Disaster Monitoring Platform
# ============================================================

# ── Standard Library ────────────────────────────────────────
import json                        # parse / save JSON responses
import hashlib                     # MD5 hash of raw API response
from datetime import datetime, timezone, timedelta   # date math & UTC timestamps
from pathlib import Path           # cross-platform file paths

# ── HTTP ────────────────────────────────────────────────────
import requests                    # fetch live data from USGS API

# ── Data Manipulation ───────────────────────────────────────
import pandas as pd                # DataFrames, cleaning, feature engineering
import numpy as np                 # numerical ops (used in type coercion)

# ── Output Formats ──────────────────────────────────────────
import pyarrow                     # required by pandas to write .parquet files

# ── Display helpers (Jupyter only) ──────────────────────────
from IPython.display import display, HTML

# ── Version check (quick sanity) ────────────────────────────
print("✅ All libraries imported successfully")
print(f"   pandas  : {pd.__version__}")
print(f"   numpy   : {np.__version__}")
print(f"   pyarrow : {pyarrow.__version__}")
print(f"   requests: {requests.__version__}")

✅ All libraries imported successfully
   pandas  : 2.2.2
   numpy   : 2.0.2
   pyarrow : 18.1.0
   requests: 2.32.4


In [2]:
END_DATE   = datetime.now(timezone.utc).date()
START_DATE = END_DATE - timedelta(days=30)

API_URL = (
    "https://earthquake.usgs.gov/fdsnws/event/1/query"
    f"?format=geojson"
    f"&starttime={START_DATE}"
    f"&endtime={END_DATE}"
    f"&minmagnitude=2.5"
    f"&orderby=time"
)

print(f"Start Date : {START_DATE}")
print(f"End Date   : {END_DATE}")
print(f"API URL    : {API_URL}")

Start Date : 2026-04-27
End Date   : 2026-05-27
API URL    : https://earthquake.usgs.gov/fdsnws/event/1/query?format=geojson&starttime=2026-04-27&endtime=2026-05-27&minmagnitude=2.5&orderby=time


In [3]:
response = requests.get(API_URL, timeout=60)
response.raise_for_status()
raw_data = response.json()

data_fetched_at_utc = datetime.now(timezone.utc).isoformat()

with open("raw_earthquakes.json", "w") as f:
    json.dump(raw_data, f, indent=2)

raw_data_hash = hashlib.md5(
    json.dumps(raw_data, sort_keys=True).encode()
).hexdigest()

raw_record_count = len(raw_data["features"])

print(f"Data Fetched At (UTC) : {data_fetched_at_utc}")
print(f"Total Records Fetched : {raw_record_count}")
print(f"Raw JSON Saved        : raw_earthquakes.json")
print(f"MD5 Hash              : {raw_data_hash}")

Data Fetched At (UTC) : 2026-05-27T09:04:16.704487+00:00
Total Records Fetched : 1785
Raw JSON Saved        : raw_earthquakes.json
MD5 Hash              : f6e7d84574b9f071552949260278f60f


In [4]:
records = []

for feature in raw_data["features"]:
    props  = feature.get("properties", {})
    coords = feature.get("geometry", {}).get("coordinates", [None, None, None])

    records.append({
        "event_id"    : feature.get("id"),
        "magnitude"   : props.get("mag"),
        "place"       : props.get("place"),
        "event_time"  : props.get("time"),
        "updated_time": props.get("updated"),
        "tsunami"     : props.get("tsunami"),
        "significance": props.get("sig"),
        "alert"       : props.get("alert"),
        "status"      : props.get("status"),
        "event_type"  : props.get("type"),
        "longitude"   : coords[0],
        "latitude"    : coords[1],
        "depth_km"    : coords[2],
    })

df = pd.DataFrame(records)

print(f"DataFrame Shape : {df.shape}")
print(f"Columns         : {list(df.columns)}")
df.head()

DataFrame Shape : (1785, 13)
Columns         : ['event_id', 'magnitude', 'place', 'event_time', 'updated_time', 'tsunami', 'significance', 'alert', 'status', 'event_type', 'longitude', 'latitude', 'depth_km']


,event_id,magnitude,place,event_time,updated_time,tsunami,significance,alert,status,event_type,longitude,latitude,depth_km
0,aka2026kjqura,2.9,"99 km W of Akhiok, Alaska",1779839286884,1779847679040,0,129,None,reviewed,earthquake,-155.8040,57.0080,59.600
1,aka2026kjpaly,2.6,"122 km WSW of Adak, Alaska",1779836089626,1779841187663,0,104,None,reviewed,earthquake,-178.1590,51.3210,9.600
2,us7000snt1,3.3,"18 km NE of Las Terrenas, Dominican Republic",1779833445686,1779857396040,0,168,None,reviewed,earthquake,-69.4309,19.4376,10.000
3,us7000snqj,3.0,"162 km WSW of Adak, Alaska",1779832596730,1779847311040,0,138,None,reviewed,earthquake,-178.9338,51.5567,83.389
4,us7000snps,4.9,"212 km ENE of Levuka, Fiji",1779827710456,1779831213040,0,369,None,reviewed,earthquake,-178.7628,-17.5166,535.844


In [5]:
missing_before = df.isnull().sum().to_dict()

print("Missing Values BEFORE Cleaning:")
print(pd.Series(missing_before))

df.dropna(subset=["event_id", "magnitude", "latitude", "longitude", "depth_km"], inplace=True)

df["place"] = df["place"].fillna("Unknown")
df["alert"] = df["alert"].fillna("none")

missing_after_drop = df.isnull().sum().to_dict()

print(f"\nRows after dropping critical nulls : {len(df)}")
df.head()

Missing Values BEFORE Cleaning:
event_id           0
magnitude          0
place              0
event_time         0
updated_time       0
tsunami            0
significance       0
alert           1721
status             0
event_type         0
longitude          0
latitude           0
depth_km           0
dtype: int64

Rows after dropping critical nulls : 1785


,event_id,magnitude,place,event_time,updated_time,tsunami,significance,alert,status,event_type,longitude,latitude,depth_km
0,aka2026kjqura,2.9,"99 km W of Akhiok, Alaska",1779839286884,1779847679040,0,129,none,reviewed,earthquake,-155.8040,57.0080,59.600
1,aka2026kjpaly,2.6,"122 km WSW of Adak, Alaska",1779836089626,1779841187663,0,104,none,reviewed,earthquake,-178.1590,51.3210,9.600
2,us7000snt1,3.3,"18 km NE of Las Terrenas, Dominican Republic",1779833445686,1779857396040,0,168,none,reviewed,earthquake,-69.4309,19.4376,10.000
3,us7000snqj,3.0,"162 km WSW of Adak, Alaska",1779832596730,1779847311040,0,138,none,reviewed,earthquake,-178.9338,51.5567,83.389
4,us7000snps,4.9,"212 km ENE of Levuka, Fiji",1779827710456,1779831213040,0,369,none,reviewed,earthquake,-178.7628,-17.5166,535.844


In [6]:
before_dedup = len(df)

df.drop_duplicates(subset=["event_id"], inplace=True)

duplicates_removed = before_dedup - len(df)

print(f"Records Before Dedup : {before_dedup}")
print(f"Duplicates Removed   : {duplicates_removed}")
print(f"Records After Dedup  : {len(df)}")

Records Before Dedup : 1785
Duplicates Removed   : 0
Records After Dedup  : 1785


In [7]:
df["event_id"]     = df["event_id"].astype(str)
df["magnitude"]    = pd.to_numeric(df["magnitude"],    errors="coerce")
df["place"]        = df["place"].astype(str)
df["event_time"]   = pd.to_datetime(df["event_time"],   unit="ms", errors="coerce")
df["updated_time"] = pd.to_datetime(df["updated_time"], unit="ms", errors="coerce")
df["tsunami"]      = pd.to_numeric(df["tsunami"],      errors="coerce").fillna(0).astype(int)
df["significance"] = pd.to_numeric(df["significance"], errors="coerce").fillna(0).astype(int)
df["alert"]        = df["alert"].astype(str)
df["status"]       = df["status"].astype(str)
df["event_type"]   = df["event_type"].astype(str)
df["longitude"]    = pd.to_numeric(df["longitude"],    errors="coerce")
df["latitude"]     = pd.to_numeric(df["latitude"],     errors="coerce")
df["depth_km"]     = pd.to_numeric(df["depth_km"],     errors="coerce")

print("Data Types After Conversion:")
print(df.dtypes)

Data Types After Conversion:
event_id                object
magnitude              float64
place                   object
event_time      datetime64[ns]
updated_time    datetime64[ns]
tsunami                  int64
significance             int64
alert                   object
status                  object
event_type              object
longitude              float64
latitude               float64
depth_km               float64
dtype: object


In [8]:
for col in ["place", "alert", "status", "event_type"]:
    df[col] = df[col].str.strip().str.lower()

df["place_clean"] = df["place"].str.title()

print("Text Cleaning Done")
print(f"\nSample place       : {df['place'].iloc[0]}")
print(f"Sample place_clean : {df['place_clean'].iloc[0]}")
print(f"Sample alert       : {df['alert'].iloc[0]}")
print(f"Sample status      : {df['status'].iloc[0]}")
print(f"Sample event_type  : {df['event_type'].iloc[0]}")
df[["place", "place_clean", "alert", "status", "event_type"]].head()

Text Cleaning Done

Sample place       : 99 km w of akhiok, alaska
Sample place_clean : 99 Km W Of Akhiok, Alaska
Sample alert       : none
Sample status      : reviewed
Sample event_type  : earthquake


,place,place_clean,alert,status,event_type
0,"99 km w of akhiok, alaska","99 Km W Of Akhiok, Alaska",none,reviewed,earthquake
1,"122 km wsw of adak, alaska","122 Km Wsw Of Adak, Alaska",none,reviewed,earthquake
2,"18 km ne of las terrenas, dominican republic","18 Km Ne Of Las Terrenas, Dominican Republic",none,reviewed,earthquake
3,"162 km wsw of adak, alaska","162 Km Wsw Of Adak, Alaska",none,reviewed,earthquake
4,"212 km ene of levuka, fiji","212 Km Ene Of Levuka, Fiji",none,reviewed,earthquake


In [9]:
df["event_year"]  = df["event_time"].dt.year
df["event_month"] = df["event_time"].dt.month
df["event_day"]   = df["event_time"].dt.day
df["event_hour"]  = df["event_time"].dt.hour

print("Date Features Created:")
df[["event_time", "event_year", "event_month", "event_day", "event_hour"]].head()

Date Features Created:


,event_time,event_year,event_month,event_day,event_hour
0,2026-05-26 23:48:06.884,2026,5,26,23
1,2026-05-26 22:54:49.626,2026,5,26,22
2,2026-05-26 22:10:45.686,2026,5,26,22
3,2026-05-26 21:56:36.730,2026,5,26,21
4,2026-05-26 20:35:10.456,2026,5,26,20


In [10]:
high_alert_vals = ["yellow", "orange", "red"]

def assign_risk(row):
    if (row["magnitude"] >= 6.0
            or row["tsunami"] == 1
            or row["alert"] in high_alert_vals):
        return "high"
    elif (row["magnitude"] >= 4.5
            or row["significance"] >= 600):
        return "medium"
    return "low"

df["risk_level"] = df.apply(assign_risk, axis=1)

print("Risk Level Distribution:")
print(df["risk_level"].value_counts())
df[["magnitude", "tsunami", "alert", "significance", "risk_level"]].head(10)

Risk Level Distribution:
risk_level
low       1377
medium     399
high         9
Name: count, dtype: int64


,magnitude,tsunami,alert,significance,risk_level
0,2.9,0,none,129,low
1,2.6,0,none,104,low
2,3.3,0,none,168,low
3,3.0,0,none,138,low
4,4.9,0,none,369,medium
5,4.5,0,none,312,medium
6,4.3,0,none,284,low
7,4.5,0,none,312,medium
8,2.6,0,none,104,low
9,5.0,0,none,385,medium


In [11]:
def assign_depth_cat(depth):
    if pd.isna(depth):
        return "Unknown"
    if depth < 70:
        return "Shallow"
    elif depth <= 300:
        return "Intermediate"
    return "Deep"

df["depth_category"] = df["depth_km"].apply(assign_depth_cat)

print("Depth Category Distribution:")
print(df["depth_category"].value_counts())
df[["depth_km", "depth_category"]].head(10)

Depth Category Distribution:
depth_category
Shallow         1423
Intermediate     294
Deep              68
Name: count, dtype: int64


,depth_km,depth_category
0,59.600,Shallow
1,9.600,Shallow
2,10.000,Shallow
3,83.389,Intermediate
4,535.844,Deep
5,62.215,Shallow
6,10.000,Shallow
7,28.219,Shallow
8,113.400,Intermediate
9,118.289,Intermediate


In [12]:
def assign_depth_cat(depth):
    if pd.isna(depth):
        return "Unknown"
    if depth < 70:
        return "Shallow"
    elif depth <= 300:
        return "Intermediate"
    return "Deep"

df["depth_category"] = df["depth_km"].apply(assign_depth_cat)

print("Depth Category Distribution:")
print(df["depth_category"].value_counts())
df[["depth_km", "depth_category"]].head(10)

Depth Category Distribution:
depth_category
Shallow         1423
Intermediate     294
Deep              68
Name: count, dtype: int64


,depth_km,depth_category
0,59.600,Shallow
1,9.600,Shallow
2,10.000,Shallow
3,83.389,Intermediate
4,535.844,Deep
5,62.215,Shallow
6,10.000,Shallow
7,28.219,Shallow
8,113.400,Intermediate
9,118.289,Intermediate


In [13]:
def extract_region(place):
    if pd.isna(place) or "," not in str(place):
        return "Unknown"
    return place.split(",")[-1].strip().title()

df["region"] = df["place"].apply(extract_region)

print("Top 10 Regions by Event Count:")
print(df["region"].value_counts().head(10))
df[["place", "region"]].head(10)

Top 10 Regions by Event Count:
region
Alaska                 563
Unknown                131
Ca                     126
Puerto Rico             79
Indonesia               67
Japan                   64
Nevada                  64
Chile                   54
Papua New Guinea        53
U.S. Virgin Islands     52
Name: count, dtype: int64


,place,region
0,"99 km w of akhiok, alaska",Alaska
1,"122 km wsw of adak, alaska",Alaska
2,"18 km ne of las terrenas, dominican republic",Dominican Republic
3,"162 km wsw of adak, alaska",Alaska
4,"212 km ene of levuka, fiji",Fiji
5,"31 km se of ōfunato, japan",Japan
6,owen fracture zone region,Unknown
7,"59 km sw of langsa, indonesia",Indonesia
8,"38 km sw of skwentna, alaska",Alaska
9,"152 km ssw of hihifo, tonga",Tonga


In [14]:
df.to_csv("earthquakes_clean.csv", index=False)
df.to_parquet("earthquakes_clean.parquet", index=False)

print("Files Saved Successfully")
print(f"CSV     : earthquakes_clean.csv     — {df.shape[0]} rows x {df.shape[1]} cols")
print(f"Parquet : earthquakes_clean.parquet — {df.shape[0]} rows x {df.shape[1]} cols")

Files Saved Successfully
CSV     : earthquakes_clean.csv     — 1785 rows x 21 cols
Parquet : earthquakes_clean.parquet — 1785 rows x 21 cols


In [15]:
missing_after = df.isnull().sum().to_dict()

top_5_regions = df["region"].value_counts().head(5).to_dict()

quality_report = {
    "data_fetched_at_utc"         : data_fetched_at_utc,
    "raw_record_count"            : raw_record_count,
    "clean_record_count"          : len(df),
    "duplicates_removed"          : duplicates_removed,
    "missing_values_before"       : {k: int(v) for k, v in missing_before.items()},
    "missing_values_after"        : {k: int(v) for k, v in missing_after.items()},
    "high_risk_events"            : int((df["risk_level"] == "high").sum()),
    "medium_risk_events"          : int((df["risk_level"] == "medium").sum()),
    "low_risk_events"             : int((df["risk_level"] == "low").sum()),
    "max_magnitude"               : float(df["magnitude"].max()),
    "deepest_earthquake_km"       : float(df["depth_km"].max()),
    "top_5_regions_by_event_count": top_5_regions,
}

with open("data_quality_report.json", "w") as f:
    json.dump(quality_report, f, indent=2)

print("Data Quality Report:")
for k, v in quality_report.items():
    print(f"  {k:35s} : {v}")

Data Quality Report:
  data_fetched_at_utc                 : 2026-05-27T09:04:16.704487+00:00
  raw_record_count                    : 1785
  clean_record_count                  : 1785
  duplicates_removed                  : 0
  missing_values_before               : {'event_id': 0, 'magnitude': 0, 'place': 0, 'event_time': 0, 'updated_time': 0, 'tsunami': 0, 'significance': 0, 'alert': 1721, 'status': 0, 'event_type': 0, 'longitude': 0, 'latitude': 0, 'depth_km': 0}
  missing_values_after                : {'event_id': 0, 'magnitude': 0, 'place': 0, 'event_time': 0, 'updated_time': 0, 'tsunami': 0, 'significance': 0, 'alert': 0, 'status': 0, 'event_type': 0, 'longitude': 0, 'latitude': 0, 'depth_km': 0, 'place_clean': 0, 'event_year': 0, 'event_month': 0, 'event_day': 0, 'event_hour': 0, 'risk_level': 0, 'depth_category': 0, 'region': 0}
  high_risk_events                    : 9
  medium_risk_events                  : 399
  low_risk_events                     : 1377
  max_magnitude      

In [17]:
print("=" * 55)
print("  ANALYSIS QUESTIONS")
print("=" * 55)

q1_region = df["region"].value_counts().idxmax()
q1_count  = df["region"].value_counts().max()
print(f"\nQ1. Region with most earthquakes")
print(f"    → {q1_region} ({q1_count} events)")

q2_row = df.loc[df["magnitude"].idxmax()]
print(f"\nQ2. Strongest earthquake")
print(f"    → Magnitude : {q2_row['magnitude']}")
print(f"    → Place     : {q2_row['place_clean']}")
print(f"    → Time      : {q2_row['event_time']}")

q3_count = int((df["risk_level"] == "high").sum())
print(f"\nQ3. High-risk events detected")
print(f"    → {q3_count} events")

print(f"\nQ4. Depth category breakdown")
for cat in ["Shallow", "Intermediate", "Deep"]:
    print(f"    → {cat:15s}: {(df['depth_category'] == cat).sum()}")

q5_day   = df.groupby(df["event_time"].dt.date).size().idxmax()
q5_count = df.groupby(df["event_time"].dt.date).size().max()
print(f"\nQ5. Day with most earthquake activity")
print(f"    → {q5_day} ({q5_count} events)")

print("\n" + "=" * 55)


  ANALYSIS QUESTIONS

Q1. Region with most earthquakes
    → Alaska (563 events)

Q2. Strongest earthquake
    → Magnitude : 6.9
    → Place     : 29 Km Ene Of Calama, Chile
    → Time      : 2026-05-25 21:52:20.043000

Q3. High-risk events detected
    → 9 events

Q4. Depth category breakdown
    → Shallow        : 1423
    → Intermediate   : 294
    → Deep           : 68

Q5. Day with most earthquake activity
    → 2026-05-10 (109 events)



In [18]:
print("=" * 55)
print("  PIPELINE COMPLETE — FINAL SUMMARY")
print("=" * 55)

print(f"\n  Raw Records          : {raw_record_count}")
print(f"  Clean Records        : {len(df)}")
print(f"  Duplicates Removed   : {duplicates_removed}")
print(f"  Max Magnitude        : {quality_report['max_magnitude']}")
print(f"  Deepest Earthquake   : {quality_report['deepest_earthquake_km']} km")
print(f"  High Risk Events     : {quality_report['high_risk_events']}")
print(f"  Medium Risk Events   : {quality_report['medium_risk_events']}")
print(f"  Low Risk Events      : {quality_report['low_risk_events']}")

print(f"\n  MD5 Hash of Raw API Response:")
print(f"  {raw_data_hash}")

print(f"\n  Output Files:")
print(f"  • raw_earthquakes.json")
print(f"  • earthquakes_clean.csv")
print(f"  • earthquakes_clean.parquet")
print(f"  • data_quality_report.json")

print("\n  Final DataFrame Preview:")
display(df[["event_id", "magnitude", "place_clean", "event_time",
            "depth_km", "depth_category", "risk_level", "region"]].head(10))

  PIPELINE COMPLETE — FINAL SUMMARY

  Raw Records          : 1785
  Clean Records        : 1785
  Duplicates Removed   : 0
  Max Magnitude        : 6.9
  Deepest Earthquake   : 639.858 km
  High Risk Events     : 9
  Medium Risk Events   : 399
  Low Risk Events      : 1377

  MD5 Hash of Raw API Response:
  f6e7d84574b9f071552949260278f60f

  Output Files:
  • raw_earthquakes.json
  • earthquakes_clean.csv
  • earthquakes_clean.parquet
  • data_quality_report.json

  Final DataFrame Preview:


,event_id,magnitude,place_clean,event_time,depth_km,depth_category,risk_level,region
0,aka2026kjqura,2.9,"99 Km W Of Akhiok, Alaska",2026-05-26 23:48:06.884,59.600,Shallow,low,Alaska
1,aka2026kjpaly,2.6,"122 Km Wsw Of Adak, Alaska",2026-05-26 22:54:49.626,9.600,Shallow,low,Alaska
2,us7000snt1,3.3,"18 Km Ne Of Las Terrenas, Dominican Republic",2026-05-26 22:10:45.686,10.000,Shallow,low,Dominican Republic
3,us7000snqj,3.0,"162 Km Wsw Of Adak, Alaska",2026-05-26 21:56:36.730,83.389,Intermediate,low,Alaska
4,us7000snps,4.9,"212 Km Ene Of Levuka, Fiji",2026-05-26 20:35:10.456,535.844,Deep,medium,Fiji
5,us7000snpq,4.5,"31 Km Se Of Ōfunato, Japan",2026-05-26 20:21:01.985,62.215,Shallow,medium,Japan
6,us7000snpn,4.3,Owen Fracture Zone Region,2026-05-26 20:04:55.636,10.000,Shallow,low,Unknown
7,us7000snpl,4.5,"59 Km Sw Of Langsa, Indonesia",2026-05-26 19:49:07.929,28.219,Shallow,medium,Indonesia
8,aka2026kjilgc,2.6,"38 Km Sw Of Skwentna, Alaska",2026-05-26 19:37:16.072,113.400,Intermediate,low,Alaska
9,us7000snpf,5.0,"152 Km Ssw Of Hihifo, Tonga",2026-05-26 19:25:11.739,118.289,Intermediate,medium,Tonga



| Cell | Task |
|---|---|
| 1 | Library imports |
| 2 | API config & URL builder |
| 3 | Fetch raw data + save JSON + MD5 |
| 4 | Flatten nested JSON → DataFrame |
| 5 | Missing value handling |
| 6 | Remove duplicates |
| 7 | Data type conversion |
| 8 | Text cleaning |
| 9 | Date feature extraction |
| 10 | Risk level engineering |
| 11 | Depth category engineering |
| 12 | Region extraction |
| 13 | Save CSV & Parquet |
| 14 | Data quality report |
| 15 | Analysis questions |
| 16 | Final summary + MD5 hash |